In [ ]:
# =============================================================================
# jobs/19_constituency_flood_situation.py
# One row per constituency: the forecast, the warnings and the MP, together.
#
# 13 starts from warnings already issued (reactive). 14 starts from the forecast
# (proactive). Each has many rows per constituency. This notebook folds both
# down to ONE row per constituency, so a dashboard, a briefing or a quick query
# needs no joins or ranking.
#
# What this script does:
#   1. Starts from every constituency (parliamentary_constituencies), so a
#      constituency with nothing flagged still has a row.
#   2. Folds fgs_constituency_mp_contacts (14) to the worst forecast risk over
#      the five days, the first day that risk occurs, the flood sources behind
#      it, and the worst risk on each of days 1 to 5.
#   3. Folds flood_warning_constituency_contacts (13) to counts of severe
#      flood warnings, flood warnings and flood alerts in force.
#   4. Flags whether the forecast escalated against the previous statement
#      (fgs_constituency_changes, from 18).
#   5. Attaches the sitting MP and their contact details.
#   6. Overwrites constituency_flood_situation in full.
#
# Ranks: 1 Very Low, 2 Low, 3 Medium, 4 High, 0 nothing flagged.
# The forecast and the warnings are kept in separate columns on purpose. They
# measure different things (a forecast of what may happen, a notice of what the
# EA expects), so this table does not merge them into one score.
#
# NOT YET INCLUDED: observed river and sea levels from the gauges. They need the
# readings and thresholds tables (notebooks 15 to 17) turned into events first.
#
# NO SPATIAL WORK HERE. Plain Spark SQL. Run it after 13 and 14.
#
# Exit values: "success".
# =============================================================================

import os
import sys

# Resolve the project root from this notebook's own location, so nothing
# depends on one person's workspace folder.
_nb_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
if not _nb_path.startswith("/Workspace"):
    _nb_path = "/Workspace" + _nb_path
PROJECT_ROOT = os.path.dirname(os.path.dirname(_nb_path))
sys.path.insert(0, PROJECT_ROOT)

from config import (
    TBL_STATEMENTS, TBL_FGS_INTERSECT_LOG, TBL_CONSTITUENCIES, TBL_MP_CONTACTS,
    TBL_FGS_CONTACTS, TBL_WARNING_CONTACTS, TBL_FGS_CHANGES, TBL_CONST_SITUATION,
    FGS_GEOMETRY_SOURCE
)
from utils.helpers import (
    get_spark, utc_now, table_exists, constituency_name_key, source_predicate,
    latest_complete_statements, risk_level_sql
)


# =============================================================================
# SETUP
# =============================================================================

spark   = get_spark()
now_iso = utc_now().isoformat()
name_key = constituency_name_key

for table in (TBL_STATEMENTS, TBL_FGS_INTERSECT_LOG, TBL_CONSTITUENCIES,
              TBL_MP_CONTACTS, TBL_FGS_CONTACTS, TBL_WARNING_CONTACTS):
    if not table_exists(spark, table):
        raise Exception(
            f"{table} does not exist. Run 04, 05, 07, 13 and 14 first."
        )

complete = latest_complete_statements(
    spark, TBL_STATEMENTS, TBL_FGS_INTERSECT_LOG, FGS_GEOMETRY_SOURCE, n=1
)
if not complete:
    raise Exception(
        f"No statement has up-to-date intersections for geometry source "
        f"'{FGS_GEOMETRY_SOURCE}'. Run 07_compute_intersections first."
    )
statement_id = int(complete[0]["statement_id"])
issued_at    = (complete[0]["issued_at"] or "").replace("'", "''")

held = {r["statement_id"] for r in
        spark.table(TBL_FGS_CONTACTS).select("statement_id").distinct().collect()}
if held and held != {statement_id}:
    print(f"WARNING: fgs_constituency_mp_contacts holds statement {sorted(held)} but the "
          f"latest complete statement is {statement_id}. Run 14 first, or the forecast "
          f"columns below are one statement behind.")

n_const = spark.table(TBL_CONSTITUENCIES).count()
if n_const == 0:
    raise Exception(f"{TBL_CONSTITUENCIES} is empty. Run 05_refresh_boundaries first.")
n_distinct = spark.table(TBL_CONSTITUENCIES).select("constituency_id").distinct().count()
if n_distinct != n_const:
    # Every other side of the joins below is unique per constituency, so a
    # duplicate here would publish two rows for one constituency.
    raise Exception(
        f"{TBL_CONSTITUENCIES} holds {n_const} rows but {n_distinct} distinct "
        f"constituency_id values. It has a duplicate constituency_id."
    )
print(f"Statement {statement_id} (issued {issued_at}); {n_const} constituencies.")


In [ ]:
# =============================================================================
# BUILD THE TABLE
# =============================================================================

have_changes = table_exists(spark, TBL_FGS_CHANGES)
if have_changes:
    c_in_source = source_predicate(spark, TBL_FGS_CHANGES, FGS_GEOMETRY_SOURCE, alias="c")
    changes_cte = f"""
    chg AS (
        SELECT c.constituency_id,
               COUNT(*)                                                         AS n_changes,
               MAX(CASE WHEN c.change_type IN ('escalated', 'new') THEN 1 ELSE 0 END) AS escalated
        FROM {TBL_FGS_CHANGES} c
        WHERE c.statement_id = {statement_id} AND {c_in_source}
        GROUP BY c.constituency_id
    )"""
else:
    print("fgs_constituency_changes does not exist yet (18 has not run): "
          "fgs_changed and fgs_escalated will be false.")
    changes_cte = """
    chg AS (
        SELECT CAST(NULL AS STRING) AS constituency_id, 0 AS n_changes, 0 AS escalated
        WHERE 1 = 0
    )"""

query = f"""
WITH mp AS (
    -- One MP per constituency key, so a duplicate name cannot fan rows out.
    SELECT * FROM (
        SELECT
            m.*,
            {name_key('m.constituency_name')} AS name_key,
            ROW_NUMBER() OVER (
                PARTITION BY {name_key('m.constituency_name')}
                ORDER BY m.member_id
            ) AS rn
        FROM {TBL_MP_CONTACTS} m
    ) ranked
    WHERE rn = 1
),
cd AS (
    -- worst rank per constituency, forecast day and flood source
    SELECT constituency_id, forecast_date, day_index, source,
           MAX(risk_rank) AS rk
    FROM {TBL_FGS_CONTACTS}
    WHERE risk_rank > 0
    GROUP BY constituency_id, forecast_date, day_index, source
),
w AS (
    SELECT constituency_id, MAX(rk) AS worst_rank FROM cd GROUP BY constituency_id
),
fgs AS (
    SELECT
        cd.constituency_id,
        w.worst_rank                                                        AS worst_rank,
        MIN(CASE WHEN cd.rk = w.worst_rank THEN cd.forecast_date END)       AS worst_date,
        concat_ws(', ', sort_array(collect_set(
            CASE WHEN cd.rk = w.worst_rank THEN cd.source END)))            AS worst_sources,
        MAX(CASE WHEN cd.day_index = 1 THEN cd.rk END)                      AS rank_day1,
        MAX(CASE WHEN cd.day_index = 2 THEN cd.rk END)                      AS rank_day2,
        MAX(CASE WHEN cd.day_index = 3 THEN cd.rk END)                      AS rank_day3,
        MAX(CASE WHEN cd.day_index = 4 THEN cd.rk END)                      AS rank_day4,
        MAX(CASE WHEN cd.day_index = 5 THEN cd.rk END)                      AS rank_day5
    FROM cd JOIN w ON w.constituency_id = cd.constituency_id
    GROUP BY cd.constituency_id, w.worst_rank
),
wn AS (
    -- warnings in force, counted once per EA area even if it spans constituencies
    SELECT constituency_id,
           COUNT(DISTINCT CASE WHEN severity_level = 1 THEN flood_area_id END) AS severe_warnings,
           COUNT(DISTINCT CASE WHEN severity_level = 2 THEN flood_area_id END) AS warnings,
           COUNT(DISTINCT CASE WHEN severity_level = 3 THEN flood_area_id END) AS alerts
    FROM {TBL_WARNING_CONTACTS}
    WHERE is_in_force
    GROUP BY constituency_id
),
{changes_cte}
SELECT
    c.constituency_id,
    c.name                                              AS constituency_name,
    -- the MP
    (mp.member_id IS NOT NULL)                          AS mp_matched,
    mp.member_id                                        AS mp_member_id,
    mp.name                                             AS mp_name,
    mp.party                                            AS mp_party,
    mp.email                                            AS mp_email,
    mp.phone                                            AS mp_phone,
    mp.website                                          AS mp_website,
    -- the forecast (proactive)
    {statement_id}                                      AS fgs_statement_id,
    '{issued_at}'                                       AS fgs_issued_at,
    COALESCE(fgs.worst_rank, 0)                         AS fgs_worst_rank,
    COALESCE({risk_level_sql('fgs.worst_rank')}, 'None') AS fgs_worst_level,
    fgs.worst_date                                      AS fgs_worst_date,
    fgs.worst_sources                                   AS fgs_worst_sources,
    COALESCE(fgs.rank_day1, 0)                          AS fgs_rank_day1,
    COALESCE(fgs.rank_day2, 0)                          AS fgs_rank_day2,
    COALESCE(fgs.rank_day3, 0)                          AS fgs_rank_day3,
    COALESCE(fgs.rank_day4, 0)                          AS fgs_rank_day4,
    COALESCE(fgs.rank_day5, 0)                          AS fgs_rank_day5,
    (COALESCE(chg.n_changes, 0) > 0)                    AS fgs_changed,
    (COALESCE(chg.escalated, 0) = 1)                    AS fgs_escalated,
    -- the warnings (reactive)
    COALESCE(wn.severe_warnings, 0)                     AS severe_warnings,
    COALESCE(wn.warnings, 0)                            AS warnings,
    COALESCE(wn.alerts, 0)                              AS alerts,
    (COALESCE(wn.severe_warnings, 0) + COALESCE(wn.warnings, 0)
        + COALESCE(wn.alerts, 0) > 0)                   AS has_active_warning,
    '{now_iso}'                                         AS computed_at
FROM {TBL_CONSTITUENCIES} c
LEFT JOIN mp  ON mp.name_key = {name_key('c.name')}
LEFT JOIN fgs ON fgs.constituency_id = c.constituency_id
LEFT JOIN wn  ON wn.constituency_id  = c.constituency_id
LEFT JOIN chg ON chg.constituency_id = c.constituency_id
"""

result = spark.sql(query).cache()
n_rows = result.count()

if n_rows != n_const:
    raise Exception(f"Built {n_rows} rows from {n_const} constituencies: a join fanned out.")

n_fgs  = result.filter("fgs_worst_rank > 0").count()
n_warn = result.filter("has_active_warning").count()
n_esc  = result.filter("fgs_escalated").count()
print(f"Constituencies: {n_rows}")
print(f"  with a forecast risk in the next five days: {n_fgs}")
print(f"  with an escalation since the previous statement: {n_esc}")
print(f"  with a warning or alert in force: {n_warn}")

unmatched = result.filter("NOT mp_matched").count()
if unmatched == n_rows:
    raise Exception(
        "No constituency matched an MP on name. Check mp_contact_details and "
        "parliamentary_constituencies."
    )
if unmatched:
    print(f"NOTE: {unmatched} constituenc(ies) have no matching MP (vacancy, or a name that does not match).")


In [ ]:
# =============================================================================
# WRITE TO DELTA
# Full overwrite: this table is the CURRENT picture, rebuilt every poll.
# =============================================================================

(
    result.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(TBL_CONST_SITUATION)
)

spark.sql(f"""
    COMMENT ON TABLE {TBL_CONST_SITUATION} IS
    'One row per constituency: the latest FGS forecast (worst risk over five days, the
     first day it occurs, its flood sources, and the worst risk on each day), whether
     the forecast escalated since the previous statement, counts of flood warnings
     and alerts in force, and the sitting MP. Rebuilt on every poll by
     19_constituency_flood_situation from fgs_constituency_mp_contacts,
     flood_warning_constituency_contacts and fgs_constituency_changes. Ranks: 1 Very
     Low, 2 Low, 3 Medium, 4 High, 0 nothing flagged. Forecast and warnings are kept
     separate on purpose. Observed gauge levels are not included.'
""")

written = spark.table(TBL_CONST_SITUATION).count()
print(f"\nVerified: {written} rows written to {TBL_CONST_SITUATION}.")
result.unpersist()

dbutils.notebook.exit("success")
